# CV04: 64-D contrastive learning + KNN

Thesis sequence: focal WideResNet → frozen pooled features → fully connected 64-D projection → contrastive objective → training-only KNN.

The CV chapter (PDF p. 94, reference 13) cites Khosla et al. **Supervised Contrastive Learning**, so SupCon is the default. The earlier thesis equation (PDF p. 77) is also implemented as `thesis_pairwise`. This source ambiguity is preserved explicitly. The original stage hyperparameters and image-view recipe are unavailable; defaults are recorded project choices. The thesis's reported 69.1 ± 1.7% is not a result of this notebook.

Run the baseline notebook first. CV04 reuses its **focal** final checkpoints for the same manifest and folds. It does not retrain the backbone or edit the baseline files. Unknown is a third expert class; vote fractions are not calibrated probabilities.

**Colab preparation:** copy `outputs/colab_upload/parkinson_training_code_v3_cv04.zip` from the delivered project into `My Drive/ParkinsonResearch/`. This is a new archive, not the existing v2 baseline archive. The setup checks its exact checksum. The shared dataset and focal runs retain their existing Drive paths. This notebook also works locally from the repository.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True


def extract_checked(source, destination, expected_sha256):
    if not source.is_file():
        raise FileNotFoundError(f'Missing Drive file: {source}. Check the Google account and whether the code archive upload is complete.')
    local_zip = Path('/content') / source.name
    shutil.copy2(source, local_zip)
    if hashlib.sha256(local_zip.read_bytes()).hexdigest() != expected_sha256:
        raise ValueError(f'Archive checksum differs from this notebook: {source.name}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_zip) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f'Unsafe archive path: {name}')
        archive.extractall(destination)


DRIVE_DIR = None
if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/ParkinsonResearch')
    DATA_ROOT = Path('/content/ParkinsonData')
    extract_checked(
        DRIVE_DIR / 'datasets/parkinson_dataset_v1.zip', DATA_ROOT,
        '85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6',
    )
    metadata = json.loads((DATA_ROOT / 'dataset_package.json').read_text())
    for name, info in metadata['files'].items():
        target = (DATA_ROOT / name).resolve()
        if not target.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f'Unsafe dataset path: {name}')
        if hashlib.sha256(target.read_bytes()).hexdigest() != info['sha256']:
            raise ValueError(f'Dataset verification failed: {name}')
    PROJECT_DIR = Path('/content/ParkinsonAiResearch')
    extract_checked(DRIVE_DIR / 'parkinson_training_code_v3_cv04.zip', PROJECT_DIR,
                    'b8f0d5ae246c0a9f701d420d513482198a502c79c7d6c2a452d075741583d820')
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_DIR = next((p for p in candidates if (p / 'train.py').is_file()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Open this notebook from the ParkinsonAiResearch repository.')
    DATA_ROOT = PROJECT_DIR

DATASET_DIR = DATA_ROOT / 'datasets'
MANIFEST = DATASET_DIR / 'annotations/fold_manifest.csv'
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Dataset manifest not found: {MANIFEST}')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError('Select Kernel → Colab → New Colab Server, choose a GPU, then rerun setup.')
print('Code directory:', PROJECT_DIR)
print('Dataset directory:', DATASET_DIR)
print('Fold manifest:', MANIFEST)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'MPS — Apple GPU' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
      else 'CPU')
print('Ready. Configure CV04, then run training and development selection.')


Start with fold 1. After the pilot succeeds, set `FOLD_END = 25` and use the same recipe. The runner resumes completed or interrupted projections; feature extraction repeats after a runtime reset. Each projection checkpoint is small, but the existing source WideResNet checkpoints must remain available. A fresh output directory cannot replace another recipe.

Temperature candidates are selected across the requested development folds together with KNN k, distance and weighting. Use a new selection directory for each selection run. Pilot scores are explicitly incomplete; all-fold scores are development selection scores, not unbiased test estimates. Fold 22 retains an empty validation report.


In [ ]:
FOLD_START = 1
FOLD_END = 1  # change to 25 after the pilot
EPOCHS = 30
SEED = 42
CONTRASTIVE_LOSS = 'supcon'  # alternative: 'thesis_pairwise'
TEMPERATURES = [0.1]  # optional development grid: [0.07, 0.1, 0.2]
MARGINS = [1.0]  # used only for pairwise objectives
NORMALIZE = True  # set False for the literal, unnormalized pairwise comparison
K_VALUES = [1, 3, 5, 7]
DISTANCES = ['l2']
WEIGHTINGS = ['uniform', 'distance']
DEVICE = 'cuda' if torch.cuda.is_available() else 'auto'
BASELINE_DIR = (DRIVE_DIR / 'runs/baselines_shared_dataset_v1/focal/checkpoints'
                if IN_COLAB else PROJECT_DIR / 'outputs/baselines/focal/checkpoints')
BASELINE_GLOB = '*_last.pt'
RECIPE = 'supcon_v1'  # change when changing objective, normalization or optimization
OUTPUT_ROOT = (DRIVE_DIR / 'runs/cv04' / RECIPE if IN_COLAB
               else PROJECT_DIR / 'outputs/cv04' / RECIPE)
SELECTION_DIR = OUTPUT_ROOT.parent / f'{RECIPE}_selection_folds{FOLD_START:02d}-{FOLD_END:02d}'
if not BASELINE_DIR.is_dir():
    raise FileNotFoundError(f'Finish the focal baselines or correct BASELINE_DIR: {BASELINE_DIR}')
print('Source focal checkpoints:', BASELINE_DIR)
print('CV04 training output:', OUTPUT_ROOT)
print('Development selection output:', SELECTION_DIR)


Train one fresh projection per source fold and temperature/margin candidate. The runner checks all source checkpoint matches before starting. Progress prints the candidate, fold and projection epoch. Training and selection open no test images.


In [ ]:
command = [sys.executable, '-u', 'train_contrastive_knn_all_folds.py',
           '--baseline-dir', str(BASELINE_DIR), '--baseline-glob', BASELINE_GLOB,
           '--manifest', str(MANIFEST), '--root', str(DATA_ROOT),
           '--output-root', str(OUTPUT_ROOT), '--fold-start', str(FOLD_START),
           '--fold-end', str(FOLD_END), '--epochs', str(EPOCHS), '--seed', str(SEED),
           '--contrastive-loss', CONTRASTIVE_LOSS, '--embedding-dim', '64',
           '--normalize' if NORMALIZE else '--no-normalize',
           '--device', DEVICE, '--resume']
parameter = '--temperatures' if CONTRASTIVE_LOSS == 'supcon' else '--margins'
values = TEMPERATURES if CONTRASTIVE_LOSS == 'supcon' else MARGINS
subprocess.run(command + [parameter, *map(str, values)], check=True)


Select one shared configuration by pooled three-class expert macro-F1. Every candidate must have every requested final fold checkpoint. Selection verifies exact OOF drawing coverage and writes its full candidate table, image and participant reports, and frozen `foldXX/selected.pt` checkpoints. These scores were used for selection and must be labeled accordingly.


In [ ]:
command = [sys.executable, '-u', 'tune_contrastive_knn.py',
           '--runs-dir', str(OUTPUT_ROOT), '--manifest', str(MANIFEST),
           '--root', str(DATA_ROOT), '--output-dir', str(SELECTION_DIR),
           '--fold-start', str(FOLD_START), '--fold-end', str(FOLD_END),
           '--device', DEVICE, '--k-values', *map(str, K_VALUES),
           '--distances', *DISTANCES, '--weightings', *WEIGHTINGS]
subprocess.run(command, check=True)
selection = json.loads((SELECTION_DIR / 'selection.json').read_text())
report = json.loads((SELECTION_DIR / 'oof_cv04_report.json').read_text())
print('Selected configuration:', selection['selected'])
print('Complete development schedule:', selection['complete_development_schedule'])
print('Expert image metrics:', report['image_expert_metrics'])
print('Clinical participant metrics:', report['participant_clinical_metrics'])


Independently reload the first selected model and reproduce its validation predictions. This does not evaluate the held-out test partition. For a future, frozen test protocol, use the documented evaluation command with its explicit test flag. A thesis-style weighted 25-model ensemble requires a separately specified weighting protocol.


In [ ]:
checkpoint = SELECTION_DIR / f'fold{FOLD_START:02d}' / 'selected.pt'
reload_dir = SELECTION_DIR / f'reloaded_fold{FOLD_START:02d}'
subprocess.run([sys.executable, '-u', 'eval_representation.py',
                '--checkpoint', str(checkpoint), '--manifest', str(MANIFEST),
                '--root', str(DATA_ROOT), '--device', DEVICE,
                '--output-dir', str(reload_dir)], check=True)
original = checkpoint.parent / 'val_predictions.csv'
assert original.read_bytes() == (reload_dir / 'val_predictions.csv').read_bytes()
print('Independent reload reproduced the saved validation predictions.')


For new unlabeled drawings, run `predict_contrastive_knn.py --checkpoint <selected.pt> --images <drawing.png> --output-dir <new-output>` with the same manifest/root arguments. The inference path takes no expert or clinical query labels. See `README/CV04_CONTRASTIVE_KNN.md` for both objectives, defaults, provenance and test commands.
